# CS781 Phase 3 — Hierarchical RR (**Encoder** backbone) + CRF

The exact recipe that scored **0.4857** on the decoder, ported to `best_enc.pt`.
Same 128-token cap, `clean_text`, max-frequency class weights, 4 positional features,
CRF + Viterbi, no sampler, plateau averaging, 3-seed ensemble.

## What had to change for the encoder

| | decoder | encoder |
|---|---|---|
| `vocab_size` | 40,000 | **40,001** — MLM reserved one id for `[MASK]` |
| layers | 19 | **14** |
| attention | GQA, 12 Q / 4 KV, **causal** | **full MHA, 12 heads, bidirectional** |
| RoPE buffer | `max_seq_len * 2` | **`max_seq_len`** — the encoder's Phase 1 used the unscaled buffer |
| extra head weights | none | **`mlm_dense`, `mlm_norm`** must load |
| pooling | last-token `</s>` + mean | **mean of `norm(x)` + mean of the MLM transform** |

## Why the pooling changed

Last-token pooling exists because a *causal* model only has full context at the final position.
A bidirectional encoder has full context at **every** position, so there is no privileged
summary slot and mean pooling is the correct readout.

The second view fixes an asymmetry in the old encoder-vs-decoder diagnostic. The encoder's
MLM objective was `mlm_head(mlm_norm(gelu(mlm_dense(norm(x)))))`, and that diagnostic discarded
`mlm_dense`/`mlm_norm` (they appeared as *unexpected* keys) and pooled `norm(x)` — two transforms
short of what the objective actually shaped. The decoder's head is `lm_head(norm(x))` with tied
weights, so its `norm(x)` sat one tied matmul from its logits and was simply closer to
readout-ready. Here both views are pooled and concatenated, so the BiLSTM picks. **CELL 15
ablates them** so you can see which one carries the signal.

Feature width is 2x768 + 4 = **1540**, identical to the decoder run, so head capacity is matched
and the two notebooks are directly comparable.

## Fair-comparison note

Your encoder has had roughly **5x less learning signal** than the decoder — MLM at 15% masking
gives gradient at only 15% of positions, so 29,000 steps is ~71M predicted positions against the
decoder's ~352M. Whatever this scores, it is not a converged architecture comparison.

In [ ]:
# CELL 1 — ASSETS  (best_enc.pt, not best.pt)

from pathlib import Path
from google.colab import drive
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download
import os, re, sys, subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")

drive_files = []
for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in {".Trash", ".ipynb_checkpoints", "node_modules"}]
    for fn in files: drive_files.append(Path(root) / fn)

def normalized_name(p): return re.sub(r" \(\d+\)(?=\.)", "", p.name).lower()

def locate_drive_file(filename, preferred_folder=None):
    tgt = filename.lower()
    m = [p for p in drive_files if p.name.lower() == tgt or normalized_name(p) == tgt]
    if preferred_folder:
        pref = [p for p in m if preferred_folder.lower() in str(p.parent).lower()]
        if pref: m = pref
    if not m: raise FileNotFoundError(f"Could not find {filename} on Drive.")
    m.sort(key=lambda p: (p.name.lower() != tgt, len(p.parts), str(p).lower()))
    return m[0]

CKPT_PATH      = locate_drive_file("best_enc.pt")      # <-- ENCODER checkpoint
TOKENIZER_JSON = locate_drive_file("bpe_40k.json")
TOKENIZER_PY   = locate_drive_file("a1_tokenizer.py")

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = getpass("HF read token: ")
api = HfApi()
phase3 = [f for f in api.list_repo_files(repo_id=HF_REPO, repo_type="dataset",
                                          token=HF_TOKEN) if "phase_3_rr/" in f.lower()]

def dl(filename):
    tgt = filename.lower()
    m = [f for f in phase3 if Path(f).name.lower() == tgt] or \
        [f for f in phase3 if normalized_name(Path(f)).lower() == tgt]
    if "test" in tgt:          pref = [f for f in m if "/test/" in f.lower()]
    elif "label_vocab" in tgt: pref = m
    else:                      pref = [f for f in m if "/train/" in f.lower()]
    return Path(hf_hub_download(repo_id=HF_REPO, filename=(pref or m)[0],
                                repo_type="dataset", token=HF_TOKEN))

CL_TRAIN, CL_DEV = dl("CL_train.json"), dl("CL_dev.json")
IT_TRAIN, IT_DEV = dl("IT_train.json"), dl("IT_dev.json")
CL_TEST, IT_TEST = dl("CL_test.json"), dl("IT_test.json")
LABEL_VOCAB = dl("label_vocab.json")
if str(TOKENIZER_PY.parent) not in sys.path: sys.path.insert(0, str(TOKENIZER_PY.parent))
print(f"Encoder: {CKPT_PATH}")

Mounted at /content/drive
HF read token: ··········


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Encoder: /content/drive/MyDrive/best_enc.pt


In [ ]:
# CELL 2 — IMPORTS & DEVICE

import json, math, random, zipfile, time, html, unicodedata
from dataclasses import dataclass
from collections import Counter, deque
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast
from sklearn.metrics import f1_score, classification_report
from a1_tokenizer import BPETokenizer

OUTPUT_DIR = DRIVE_ROOT / "CS781_Phase3_EncoderHierCRF_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1024**3:.1f} GiB)")

In [ ]:
# CELL 3 — HYPERPARAMETERS

@dataclass
class TrainCfg:
    # ENCODER arch — must EXACTLY match Phase 1 encoder_legal_lm
    vocab_size: int = 40_001        # 40,000 BPE + 1 reserved [MASK] id
    d_model: int    = 768
    n_layers: int   = 14            # decoder had 19
    n_heads: int    = 12            # full MHA — no GQA, no n_kv_heads
    d_ff: int       = 2048
    max_seq_len: int = 512          # RoPE buffer is max_seq_len (NOT *2 — that is the decoder)
    rope_theta: float = 10_000.0
    dropout: float = 0.0            # frozen backbone in eval mode

    # Sentence encoding
    sent_max_tokens: int = 128      # p50=31, p90=65, p95=80, p99=131 tokens
    encode_batch_size: int = 64
    n_pos_feats: int = 4
    pool_views: tuple = ("norm", "mlm")   # mean of norm(x) and of the MLM transform

    # Head
    num_labels: int = 13
    bilstm_hidden: int = 256
    bilstm_layers: int = 2
    bilstm_dropout: float = 0.3
    cls_dropout: float = 0.4
    aux_ce_weight: float = 0.5
    crf_lr: float = 5e-3
    class_weight_cap: float = 4.0

    # Training
    epochs: int = 60
    lr: float = 1e-3
    weight_decay: float = 0.01
    max_grad_norm: float = 5.0
    early_stop_patience: int = 10
    swa_window: int = 8
    seed: int = 42
    DEV_NOISE: float = 0.026        # measured: the 10-doc dev split alone moves F1 this much

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_heads

CFG = TrainCfg()
print(f"Encoder: {CFG.n_layers}L, {CFG.n_heads}H bidirectional, vocab={CFG.vocab_size}")
print(f"RoPE buffer = {CFG.max_seq_len} (decoder used {CFG.max_seq_len*2})")
print(f"sent_max={CFG.sent_max_tokens}, pool_views={CFG.pool_views}")

Encoder: 14L, 12H bidirectional, vocab=40001
RoPE buffer = 512 (decoder used 1024)
sent_max=128, pool_views=('norm', 'mlm')


In [ ]:
# CELL 4 — SEED + TOKENIZER + clean_text

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed); torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)
tokenizer = BPETokenizer.load(str(TOKENIZER_JSON))
BASE_VOCAB_SIZE = int(tokenizer.vocab_size)

# The MLM [MASK] id sits immediately after the BPE vocabulary. Phase 3 never masks,
# but the embedding table must be 40,001 rows for the checkpoint to load.
MASK_ID = BASE_VOCAB_SIZE
assert CFG.vocab_size == BASE_VOCAB_SIZE + 1, (CFG.vocab_size, BASE_VOCAB_SIZE)

PAD_ID = int(tokenizer.token_to_id["<pad>"])
BOS_ID = int(tokenizer.token_to_id["<s>"])
EOS_ID = int(tokenizer.token_to_id["</s>"])
print(f"base vocab={BASE_VOCAB_SIZE}, MASK={MASK_ID}, model vocab={CFG.vocab_size}")
print(f"PAD={PAD_ID} BOS={BOS_ID} EOS={EOS_ID}")

TAG_RE=re.compile(r"</?[A-Za-z][^<>]{0,200}>"); URL_RE=re.compile(r"https?://\S+|www\.\S+", re.I)
ENT_RE=re.compile(r"&(?:nbsp|amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);", re.I)
CTRL_RE=re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
WSP_RE=re.compile(r"[\u00a0\u1680\u180e\u2000-\u200a\u202f\u205f\u3000]")
ZW_RE=re.compile(r"[\u200b\u200c\u200d\u2060]"); SPACE_RE=re.compile(r"\s+")

def clean_text(t):
    if t is None: return ""
    t = str(t)
    if t.count("\ufffd") > 50: return ""
    t = TAG_RE.sub(" ", t); t = html.unescape(t)
    t = URL_RE.sub(" ", t); t = ENT_RE.sub(" ", t)
    t = t.replace("\ufeff"," ").replace("\ufffd"," ")
    t = CTRL_RE.sub(" ", t); t = WSP_RE.sub(" ", t)
    t = unicodedata.normalize("NFKC", t); t = ZW_RE.sub("", t)
    return SPACE_RE.sub(" ", t).strip()

def encode_sentence(s):
    ids = tokenizer.encode(clean_text(s)) or [PAD_ID]
    return [BOS_ID] + ids[: CFG.sent_max_tokens - 2] + [EOS_ID]

def pos_features(n):
    i = np.arange(n, dtype=np.float32); rel = i / max(n - 1, 1)
    return np.stack([rel, rel**2, np.exp(-i/10.0),
                     np.exp(-(n-1-i)/10.0)], 1).astype(np.float32)

_t = "The Supreme Court held that the appellant was entitled to relief."
print(f"smoke test: {len(encode_sentence(_t))} tokens")

base vocab=40000, MASK=40000, model vocab=40001
PAD=0 BOS=2 EOS=3
smoke test: 13 tokens


In [ ]:
# CELL 5 — LOAD DATA + CLASS WEIGHTS

with open(LABEL_VOCAB) as f: LABEL2ID = json.load(f)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID); CFG.num_labels = NUM_LABELS
print(f"Labels ({NUM_LABELS}): {list(LABEL2ID.keys())}")

def load_lab(p):
    return [{"doc_id": d["id"], "sentences": d["text"],
             "labels": [LABEL2ID[l] for l in d["labels"]]} for d in json.load(open(p))]
def load_test(p):
    return [{"doc_id": k, "sentences": v, "labels": None} for k, v in json.load(open(p)).items()]

train_docs = load_lab(CL_TRAIN) + load_lab(IT_TRAIN)
cl_dev, it_dev = load_lab(CL_DEV), load_lab(IT_DEV)
dev_docs = cl_dev + it_dev
test_docs = load_test(CL_TEST) + load_test(IT_TEST)
for d in train_docs + dev_docs:
    assert len(d["sentences"]) == len(d["labels"]), d["doc_id"]
print(f"Train {len(train_docs)}d/{sum(len(d['sentences']) for d in train_docs)}s | "
      f"Dev {len(dev_docs)}d/{sum(len(d['sentences']) for d in dev_docs)}s | "
      f"Test {len(test_docs)}d/{sum(len(d['sentences']) for d in test_docs)}s")

cnt = Counter(l for d in train_docs for l in d["labels"])
f_max = max(cnt.values())
# Normalise against the MOST COMMON class, not the mean. The old mean was half-made-of
# PrecedentOverruled (1/sqrt(3/16845) = 74.9 of a ~147 total), which crushed 11 of 13
# weights below 1.0 and put Fact at 0.192.
class_weights = torch.tensor(
    [min(CFG.class_weight_cap, math.sqrt(f_max / max(cnt.get(i, 1), 1)))
     for i in range(NUM_LABELS)], dtype=torch.float32).to(DEVICE)
print("\nClass weights (1.0 = most common class):")
for i in sorted(range(NUM_LABELS), key=lambda i: -cnt.get(i, 0)):
    print(f"  {ID2LABEL[i]:<28} n={cnt.get(i,0):>5}  w={class_weights[i]:.3f}")

Labels (13): ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
Train 80d/16845s | Dev 10d/2142s | Test 10d/2197s

Class weights (1.0 = most common class):
  RatioOfTheDecision           n= 4301  w=1.000
  Fact                         n= 4098  w=1.024
  PrecedentReliedUpon          n= 1945  w=1.487
  ArgumentPetitioner           n= 1613  w=1.633
  ArgumentRespondent           n= 1348  w=1.786
  Statute                      n= 1099  w=1.978
  RulingByLowerCourt           n=  874  w=2.218
  PrecedentNotReliedUpon       n=  569  w=2.749
  RulingByPresentCourt         n=  307  w=3.743
  None                         n=  297  w=3.805
  Dissent                      n=  284  w=3.892
  Issue                        n=  107  w=4.000
  PrecedentOverruled           n=    3  w=4.000


In [ ]:
# CELL 6 — ENCODER ARCHITECTURE (matches Phase 1 LegalEncoderLM exactly)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    a = torch.outer(torch.arange(max_len).float(), freqs)
    return a.cos(), a.sin()

def apply_rope(x, cos, sin):
    """No offset argument — the Phase 1 encoder's apply_rope had none."""
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], dim=-1)
    ce = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], -1)
    se = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], -1)
    return x * ce + rot * se

class MultiHeadSelfAttention(nn.Module):
    """FULL bidirectional self-attention — no causal mask. Only padding is masked."""
    def __init__(self, cfg):
        super().__init__()
        self.nh = cfg.n_heads; self.hd = cfg.head_dim; self.dm = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B,T,self.nh,self.hd).transpose(1,2)
        k = self.k(x).view(B,T,self.nh,self.hd).transpose(1,2)
        v = self.v(x).view(B,T,self.nh,self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin); k = apply_rope(k, cos, sin)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if padding_mask is not None:
            w = w.masked_fill(padding_mask.unsqueeze(1).unsqueeze(2), float("-inf"))
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B,T,self.dm)
        return self.resid_drop(self.o(out))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        return x + self.ffn(self.norm2(x))

class LegalEncoderBackbone(nn.Module):
    """Phase 1 encoder. Keeps mlm_dense/mlm_norm so the MLM-aligned view can be pooled;
    mlm_head is dropped (it is tied to tok_emb and we are not predicting tokens)."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.layers  = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.d_model)
        self.mlm_dense = nn.Linear(cfg.d_model, cfg.d_model)   # has a bias
        self.mlm_norm  = RMSNorm(cfg.d_model)
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, input_ids, padding_mask=None):
        """-> (norm(x), mlm_transform(norm(x)))"""
        x = self.drop(self.tok_emb(input_ids))
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin, padding_mask)
        h = self.norm(x)
        m = self.mlm_norm(F.gelu(self.mlm_dense(h)))
        return h, m

print("Encoder architecture defined ✓ (bidirectional, no causal mask)")

Encoder architecture defined ✓ (bidirectional, no causal mask)


In [ ]:
# CELL 7 — LOAD PHASE 1 ENCODER + FREEZE

encoder = LegalEncoderBackbone(CFG).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
state = ckpt["model"] if "model" in ckpt else ckpt
print(f"Checkpoint keys sample: {list(state.keys())[:5]}")
print(f"Checkpoint: step={ckpt.get('step','?')}, val_loss={ckpt.get('val_loss','?')}")

ck_vocab, ck_d = state["tok_emb.weight"].shape
ck_layers = len({k.split('.')[1] for k in state
                 if k.startswith("layers.") and k.split('.')[1].isdigit()})
print(f"checkpoint shape: vocab={ck_vocab}, d={ck_d}, layers={ck_layers}")
assert ck_vocab == CFG.vocab_size, f"vocab mismatch {ck_vocab} vs {CFG.vocab_size}"
assert ck_layers == CFG.n_layers,  f"layer mismatch {ck_layers} vs {CFG.n_layers}"

# mlm_head is tied to tok_emb; we do not need it.
wanted = {k: v for k, v in state.items() if not k.startswith("mlm_head")}
res = encoder.load_state_dict(wanted, strict=False)
print(f"Loaded. Missing={len(res.missing_keys)} Unexpected={len(res.unexpected_keys)}")
if res.missing_keys:    print(f"  missing: {res.missing_keys[:8]}")
if res.unexpected_keys: print(f"  unexpected: {res.unexpected_keys[:8]}")
assert len(res.missing_keys) == 0, res.missing_keys
print("  mlm_dense/mlm_norm loaded — the MLM-aligned view is available.")

for p in encoder.parameters(): p.requires_grad = False
encoder.eval()
print(f"\nEncoder backbone: {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M params, frozen")

Checkpoint keys sample: ['tok_emb.weight', 'layers.0.norm1.weight', 'layers.0.attn.q.weight', 'layers.0.attn.k.weight', 'layers.0.attn.v.weight']
Checkpoint: step=29000, val_loss=1.69698847756069
checkpoint shape: vocab=40001, d=768, layers=14
Loaded. Missing=0 Unexpected=0
  mlm_dense/mlm_norm loaded — the MLM-aligned view is available.

Encoder backbone: 130.4M params, frozen


In [ ]:
# CELL 8 — CACHE SENTENCE EMBEDDINGS  (mean of norm(x) | mean of MLM transform | positional)

@torch.no_grad()
def encode_sentences(sentences, backbone, batch_size=64):
    """-> (N, 2*d_model) float32.
    MEAN pooling for both views. A bidirectional encoder has full context at every
    position, so there is no privileged summary slot the way a causal model has its
    final token — mean is the correct readout here."""
    backbone.eval()
    toks = [encode_sentence(s) for s in sentences]
    order = np.argsort([len(t) for t in toks])
    out = np.zeros((len(toks), 2 * CFG.d_model), dtype=np.float32)
    for start in range(0, len(order), batch_size):
        idxs = order[start:start+batch_size]
        chunk = [toks[i] for i in idxs]
        L = max(len(t) for t in chunk)
        ids = torch.full((len(chunk), L), PAD_ID, dtype=torch.long, device=DEVICE)
        pm  = torch.ones((len(chunk), L), dtype=torch.bool, device=DEVICE)
        for j, t in enumerate(chunk):
            ids[j, :len(t)] = torch.tensor(t, dtype=torch.long, device=DEVICE)
            pm[j, :len(t)] = False
        with autocast(device_type="cuda", dtype=torch.float16):
            h, m = backbone(ids, padding_mask=pm)
        h, m = h.float(), m.float()
        mf = (~pm).unsqueeze(-1).float(); den = mf.sum(1).clamp(min=1)
        hv = (h * mf).sum(1) / den
        mv = (m * mf).sum(1) / den
        out[idxs] = torch.cat([hv, mv], -1).cpu().numpy()
    return out

def embed_docs(docs):
    for i, d in enumerate(docs):
        e = encode_sentences(d["sentences"], encoder, CFG.encode_batch_size)
        d["emb"] = np.concatenate([e, pos_features(len(d["sentences"]))], axis=1)
        if (i+1) % 20 == 0 or i+1 == len(docs): print(f"  {i+1}/{len(docs)}")

t0 = time.time()
print("Train...");  embed_docs(train_docs)
print("Dev...");    embed_docs(dev_docs)
print("Test...");   embed_docs(test_docs)
print(f"\nEncoded in {(time.time()-t0)/60:.1f} min")

D_IN = train_docs[0]["emb"].shape[1]
# column slices, for the pooling ablation in CELL 15
SL_NORM = list(range(0, CFG.d_model))
SL_MLM  = list(range(CFG.d_model, 2*CFG.d_model))
SL_POS  = list(range(2*CFG.d_model, D_IN))
mb = sum(d["emb"].nbytes for d in train_docs+dev_docs+test_docs)/1024**2
print(f"Cached {mb:.0f} MB | D_IN={D_IN} "
      f"(= norm {CFG.d_model} + mlm {CFG.d_model} + pos {CFG.n_pos_feats})")

del encoder; torch.cuda.empty_cache()
print("Encoder freed from VRAM — every run from here trains in ~1 min.")

Train...
  20/80
  40/80
  60/80
  80/80
Dev...
  10/10
Test...
  10/10

Encoded in 0.3 min
Cached 124 MB | D_IN=1540 (= norm 768 + mlm 768 + pos 4)
Encoder freed from VRAM — every run from here trains in ~1 min.


In [ ]:
# CELL 9 — CRF + BiLSTM HEAD
#   CRF verified against brute-force enumeration of all C^T paths: logZ, Viterbi,
#   gold score and the T=1 edge case agree to ~1e-7.

class CRF(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.num_tags = C
        self.trans = nn.Parameter(torch.zeros(C, C))   # [from, to]
        self.start = nn.Parameter(torch.zeros(C))
        self.end   = nn.Parameter(torch.zeros(C))
    def _gold_score(self, emis, tags):
        T = tags.size(0)
        s = self.start[tags[0]] + self.end[tags[-1]]
        s = s + emis[torch.arange(T, device=emis.device), tags].sum()
        if T > 1: s = s + self.trans[tags[:-1], tags[1:]].sum()
        return s
    def _log_partition(self, emis):
        a = self.start + emis[0]
        for t in range(1, emis.size(0)):
            a = torch.logsumexp(a.unsqueeze(1) + self.trans, dim=0) + emis[t]
        return torch.logsumexp(a + self.end, dim=0)
    def nll(self, emis, tags):
        """Length-normalised so a 1816-sentence document does not swamp a 19-sentence one."""
        return (self._log_partition(emis) - self._gold_score(emis, tags)) / emis.size(0)
    @torch.no_grad()
    def viterbi(self, emis):
        T, C = emis.shape
        score = self.start + emis[0]; ptrs = []
        for t in range(1, T):
            best, idx = (score.unsqueeze(1) + self.trans).max(dim=0)
            score = best + emis[t]; ptrs.append(idx)
        score = score + self.end
        cur = int(score.argmax()); path = [cur]
        for idx in reversed(ptrs): cur = int(idx[cur]); path.append(cur)
        return path[::-1]

class HierarchicalCRFClassifier(nn.Module):
    def __init__(self, d_in, cfg):
        super().__init__()
        self.input_norm = nn.LayerNorm(d_in)
        self.bilstm = nn.LSTM(d_in, cfg.bilstm_hidden, cfg.bilstm_layers,
                              bidirectional=True, batch_first=True,
                              dropout=cfg.bilstm_dropout if cfg.bilstm_layers > 1 else 0.0)
        self.out_dropout = nn.Dropout(cfg.cls_dropout)
        self.head = nn.Sequential(
            nn.Linear(2*cfg.bilstm_hidden, cfg.bilstm_hidden), nn.GELU(),
            nn.Dropout(cfg.cls_dropout), nn.Linear(cfg.bilstm_hidden, cfg.num_labels))
        self.crf = CRF(cfg.num_labels)
    def emissions(self, x):
        out, _ = self.bilstm(self.input_norm(x).unsqueeze(0))
        return self.head(self.out_dropout(out[0]))

criterion = nn.CrossEntropyLoss(weight=class_weights)
macro_f1 = lambda a, b: f1_score(a, b, labels=list(range(NUM_LABELS)),
                                  average="macro", zero_division=0)
ytd = [l for d in dev_docs for l in d["labels"]]

_m = HierarchicalCRFClassifier(D_IN, CFG)
print(f"Head: {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params "
      f"(CRF: {sum(p.numel() for p in _m.crf.parameters())})")
del _m

Head: 5.40M params (CRF: 195)


In [ ]:
# CELL 10 — TRAIN / EVAL, with plateau averaging instead of one lucky epoch

def feat(d, cols=None):
    x = d["emb"] if cols is None else d["emb"][:, cols]
    return torch.from_numpy(np.ascontiguousarray(x)).to(DEVICE)

def swa_crf_from(buf):
    c = CRF(NUM_LABELS).to(DEVICE)
    with torch.no_grad():
        c.trans.copy_(torch.tensor(np.mean([a[0] for a in buf], 0), dtype=torch.float32))
        c.start.copy_(torch.tensor(np.mean([a[1] for a in buf], 0), dtype=torch.float32))
        c.end.copy_(torch.tensor(np.mean([a[2] for a in buf], 0), dtype=torch.float32))
    return c

def decode(emis_by_doc, docs, crf, use_crf=True):
    yt, yp = [], []
    for d in docs:
        e = torch.tensor(emis_by_doc[d["doc_id"]], dtype=torch.float32, device=DEVICE)
        yp.extend(crf.viterbi(e) if use_crf else e.argmax(-1).cpu().tolist())
        if d["labels"] is not None: yt.extend(d["labels"])
    return yt, yp

def run(seed, cols=None, epochs=None, verbose=False):
    """Train the head on a (possibly sliced) feature view. Returns plateau-averaged state."""
    epochs = epochs or CFG.epochs
    seed_everything(seed)
    d_in = D_IN if cols is None else len(cols)
    model = HierarchicalCRFClassifier(d_in, CFG).to(DEVICE)
    crf_ids = {id(p) for p in model.crf.parameters()}
    opt = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if id(p) not in crf_ids],
         "lr": CFG.lr, "weight_decay": CFG.weight_decay},
        {"params": list(model.crf.parameters()), "lr": CFG.crf_lr, "weight_decay": 0.0}])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    sdev, stest, scrf = (deque(maxlen=CFG.swa_window) for _ in range(3))
    best, best_ep, bad, hist = -1.0, -1, 0, []
    order = list(range(len(train_docs)))
    for ep in range(epochs):
        model.train(); random.shuffle(order)
        tl = 0.0; corr = 0; seen = 0
        for i in order:
            d = train_docs[i]
            x = feat(d, cols)
            y = torch.tensor(d["labels"], dtype=torch.long, device=DEVICE)
            e = model.emissions(x)
            loss = model.crf.nll(e, y) + CFG.aux_ce_weight * criterion(e, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm); opt.step()
            tl += float(loss.detach())
            corr += int((e.argmax(-1) == y).sum()); seen += y.numel()
        sch.step()

        model.eval()
        with torch.no_grad():
            de = {d["doc_id"]: model.emissions(feat(d, cols)).cpu().numpy() for d in dev_docs}
            te = {d["doc_id"]: model.emissions(feat(d, cols)).cpu().numpy() for d in test_docs}
        _, yp = decode(de, dev_docs, model.crf)
        f1 = macro_f1(ytd, yp)
        sdev.append(de); stest.append(te)
        scrf.append(tuple(v.detach().cpu().numpy().copy()
                          for v in (model.crf.trans, model.crf.start, model.crf.end)))
        hist.append({"epoch": ep+1, "train_loss": tl/len(order),
                     "train_acc": corr/max(seen,1), "dev_f1": f1})
        if f1 > best: best, best_ep, bad = f1, ep+1, 0
        else: bad += 1
        if verbose:
            print(f"  Ep {ep+1:2d}/{epochs} | tr_loss={tl/len(order):.4f} "
                  f"acc={corr/max(seen,1):.3f} | dev F1={f1:.4f}"
                  f"{' *' if f1 == best else ''}")
        if bad >= CFG.early_stop_patience: break

    c = swa_crf_from(scrf)
    dev_e  = {d["doc_id"]: np.mean([b[d["doc_id"]] for b in sdev], 0)  for d in dev_docs}
    test_e = {d["doc_id"]: np.mean([b[d["doc_id"]] for b in stest], 0) for d in test_docs}
    _, yp = decode(dev_e, dev_docs, c)
    return {"best": best, "best_ep": best_ep, "swa": macro_f1(ytd, yp),
            "crf": c, "dev_e": dev_e, "test_e": test_e, "hist": hist,
            "n_epochs": len(hist)}

print("Train/eval ready ✓")

Train/eval ready ✓


In [ ]:
# CELL 11 — MAIN RUN (seed 42, all views)

print("=" * 72); print("ENCODER-HIERARCHICAL + CRF"); print("=" * 72)
t0 = time.time()
main = run(CFG.seed, cols=None, verbose=True)
print(f"\n{(time.time()-t0)/60:.1f} min | best single epoch {main['best']:.4f} "
      f"@ ep {main['best_ep']} of {main['n_epochs']} | plateau-avg {main['swa']:.4f}")
if main["best_ep"] >= main["n_epochs"] - 1 and main["n_epochs"] >= CFG.epochs:
    print("WARNING: best epoch is the last one and the run hit the epoch cap — raise CFG.epochs.")
    print("This is exactly how the old encoder-vs-decoder diagnostic truncated its encoder arm.")
print(f"\nNOTE: the dev split carries a +/-{CFG.DEV_NOISE} noise floor, so do not read")
print("differences under ~0.03 as real. CELL 13 runs 3 seeds to give you a spread.")

ENCODER-HIERARCHICAL + CRF
  Ep  1/60 | tr_loss=3.0333 acc=0.280 | dev F1=0.1166 *
  Ep  2/60 | tr_loss=2.2434 acc=0.406 | dev F1=0.1797 *
  Ep  3/60 | tr_loss=1.8377 acc=0.428 | dev F1=0.2166 *
  Ep  4/60 | tr_loss=1.6786 acc=0.438 | dev F1=0.1967
  Ep  5/60 | tr_loss=1.4761 acc=0.495 | dev F1=0.3220 *
  Ep  6/60 | tr_loss=1.3622 acc=0.479 | dev F1=0.2640
  Ep  7/60 | tr_loss=1.2604 acc=0.515 | dev F1=0.2980
  Ep  8/60 | tr_loss=1.2078 acc=0.523 | dev F1=0.3255 *
  Ep  9/60 | tr_loss=1.1140 acc=0.537 | dev F1=0.3523 *
  Ep 10/60 | tr_loss=1.0922 acc=0.528 | dev F1=0.4160 *
  Ep 11/60 | tr_loss=0.9926 acc=0.563 | dev F1=0.4085
  Ep 12/60 | tr_loss=0.9422 acc=0.576 | dev F1=0.3507
  Ep 13/60 | tr_loss=0.9090 acc=0.549 | dev F1=0.3840
  Ep 14/60 | tr_loss=0.8759 acc=0.597 | dev F1=0.4078
  Ep 15/60 | tr_loss=0.7618 acc=0.633 | dev F1=0.3999
  Ep 16/60 | tr_loss=0.7334 acc=0.636 | dev F1=0.4042
  Ep 17/60 | tr_loss=0.6904 acc=0.644 | dev F1=0.3800
  Ep 18/60 | tr_loss=0.6254 acc=0.665 | d

In [ ]:
# CELL 12 — DEV REPORT + CRF decode delta + fragmentation

names = [ID2LABEL[i] for i in range(NUM_LABELS)]
_, yp_crf = decode(main["dev_e"], dev_docs, main["crf"], use_crf=True)
_, yp_arg = decode(main["dev_e"], dev_docs, main["crf"], use_crf=False)

print(f"DEV macro-F1 = {macro_f1(ytd, yp_crf):.4f}  "
      f"acc = {np.mean(np.array(ytd)==np.array(yp_crf)):.4f}\n")
print(classification_report(ytd, yp_crf, labels=list(range(NUM_LABELS)),
                            target_names=names, digits=4, zero_division=0))
sup = np.bincount(ytd, minlength=NUM_LABELS)
ok = [i for i in range(NUM_LABELS) if sup[i] > 0]
print(f"supported-only ({len(ok)}/{NUM_LABELS}): "
      f"{f1_score(ytd, yp_crf, labels=ok, average='macro', zero_division=0):.4f}")
print(f"  (Dissent and PrecedentOverruled are absent from dev, so the 13-label number")
print(f"   is mechanically capped at {len(ok)}/13 = {len(ok)/13:.4f})")

print(f"\nCRF decode: argmax {macro_f1(ytd, yp_arg):.4f} -> Viterbi "
      f"{macro_f1(ytd, yp_crf):.4f}  delta {macro_f1(ytd,yp_crf)-macro_f1(ytd,yp_arg):+.4f}")

def runlen(s):
    r, o = 1, []
    for i in range(1, len(s)):
        if s[i] == s[i-1]: r += 1
        else: o.append(r); r = 1
    o.append(r); return float(np.mean(o))
print(f"run length  gold {runlen(ytd):.2f} | argmax {runlen(yp_arg):.2f} | "
      f"Viterbi {runlen(yp_crf):.2f}   (gold pool 7.49)")
print("Viterbi far above gold => over-smoothing; lower crf_lr or scale transitions at decode.")

for tag, grp in (("CL", cl_dev), ("IT", it_dev)):
    yt_g, yp_g = decode(main["dev_e"], grp, main["crf"])
    print(f"{tag} dev: F1={macro_f1(yt_g, yp_g):.4f}")

DEV macro-F1 = 0.4516  acc = 0.6200

                        precision    recall  f1-score   support

                  Fact     0.7000    0.6937    0.6968       444
                 Issue     0.6667    0.1333    0.2222        15
    ArgumentPetitioner     0.7405    0.6350    0.6837       400
    ArgumentRespondent     0.6649    0.4596    0.5435       272
               Statute     0.8900    0.7417    0.8091       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.6190    0.5630    0.5897       254
PrecedentNotReliedUpon     0.2540    0.3200    0.2832        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.4167    0.5882    0.4878       102
    RatioOfTheDecision     0.5373    0.7320    0.6197       403
  RulingByPresentCourt     0.6341    0.5306    0.5778        49
                  None     0.4348    0.3030    0.3571        33

              accuracy                         0.6200      2142
 

In [ ]:
# CELL 13 — 3-SEED ENSEMBLE  (~3 min; embeddings are cached)

SEEDS = (42, 1337, 2024)
runs, emis_sum, crf_sum = [], None, {"t": 0.0, "s": 0.0, "e": 0.0}
dev_sum = None

for sd in SEEDS:
    r = main if sd == CFG.seed else run(sd, cols=None)
    runs.append(r)
    print(f"  seed {sd}: best {r['best']:.4f} | swa {r['swa']:.4f} "
          f"| stopped at ep {r['n_epochs']}")
    if emis_sum is None:
        emis_sum = {k: v.astype(np.float64).copy() for k, v in r["test_e"].items()}
        dev_sum  = {k: v.astype(np.float64).copy() for k, v in r["dev_e"].items()}
    else:
        for k in emis_sum: emis_sum[k] += r["test_e"][k]
        for k in dev_sum:  dev_sum[k]  += r["dev_e"][k]
    crf_sum["t"] = crf_sum["t"] + r["crf"].trans.detach().cpu().numpy()
    crf_sum["s"] = crf_sum["s"] + r["crf"].start.detach().cpu().numpy()
    crf_sum["e"] = crf_sum["e"] + r["crf"].end.detach().cpu().numpy()

swa_list = [r["swa"] for r in runs]
print(f"\nper-seed plateau F1: mean {np.mean(swa_list):.4f} +/- {np.std(swa_list):.4f}")
print("That std is the floor on any single-run A/B you might be tempted to believe.")

ens = CRF(NUM_LABELS).to(DEVICE)
with torch.no_grad():
    ens.trans.copy_(torch.tensor(crf_sum["t"]/len(SEEDS), dtype=torch.float32))
    ens.start.copy_(torch.tensor(crf_sum["s"]/len(SEEDS), dtype=torch.float32))
    ens.end.copy_(torch.tensor(crf_sum["e"]/len(SEEDS), dtype=torch.float32))
ens_dev_e  = {k: v/len(SEEDS) for k, v in dev_sum.items()}
ens_test_e = {k: v/len(SEEDS) for k, v in emis_sum.items()}
_, yp_e = decode(ens_dev_e, dev_docs, ens)
ENS_F1 = macro_f1(ytd, yp_e)
print(f"\n3-seed ENSEMBLE dev F1 = {ENS_F1:.4f}  "
      f"(best single seed {max(swa_list):.4f})")

  seed 42: best 0.4680 | swa 0.4516 | stopped at ep 35
  seed 1337: best 0.4384 | swa 0.4163 | stopped at ep 35
  seed 2024: best 0.4482 | swa 0.4245 | stopped at ep 38

per-seed plateau F1: mean 0.4308 +/- 0.0151
That std is the floor on any single-run A/B you might be tempted to believe.

3-seed ENSEMBLE dev F1 = 0.4340  (best single seed 0.4516)


In [ ]:
# CELL 14 — *** POOLING ABLATION ***  does the MLM transform carry signal?
#   This is the asymmetry the old encoder-vs-decoder diagnostic had: it pooled norm(x)
#   only and threw mlm_dense/mlm_norm away. One seed each, ~3 min total.

views = {"norm(x) only":      SL_NORM + SL_POS,
         "MLM transform only": SL_MLM + SL_POS,
         "both (default)":     None}
abl = {}
for name, cols in views.items():
    r = main if cols is None else run(CFG.seed, cols=cols)
    abl[name] = r["swa"]
    nd = D_IN if cols is None else len(cols)
    print(f"  {name:<22} d_in={nd:<5} plateau F1 = {r['swa']:.4f}")

bestv = max(abl, key=abl.get)
spread = max(abl.values()) - min(abl.values())
print(f"\nbest view: {bestv} ({abl[bestv]:.4f}), spread {spread:.4f}")
if spread < CFG.DEV_NOISE:
    print(f"  spread < the {CFG.DEV_NOISE} noise floor => the readout choice does NOT")
    print("  matter here, so it was not what cost the encoder the old diagnostic.")
else:
    print("  spread clears the noise floor => the readout choice was a real confound in")
    print("  the old diagnostic, which pooled norm(x) only.")

  norm(x) only           d_in=772   plateau F1 = 0.4446
  MLM transform only     d_in=772   plateau F1 = 0.4331
  both (default)         d_in=1540  plateau F1 = 0.4516

best view: both (default) (0.4516), spread 0.0185
  spread < the 0.026 noise floor => the readout choice does NOT
  matter here, so it was not what cost the encoder the old diagnostic.


In [ ]:
# CELL 15 — SUBMISSION (3-seed ensemble)

_, yp_test = decode(ens_test_e, test_docs, ens)
cursor = 0; predictions = {}
for d in test_docs:
    n = len(d["sentences"])
    predictions[d["doc_id"]] = [ID2LABEL[int(p)] for p in yp_test[cursor:cursor+n]]
    cursor += n
assert cursor == len(yp_test)
assert len(predictions) == len(test_docs)
for d in test_docs:
    assert len(predictions[d["doc_id"]]) == len(d["sentences"]), d["doc_id"]
assert all(l in LABEL2ID for v in predictions.values() for l in v)

pp = OUTPUT_DIR / "prediction.json"
json.dump(predictions, open(pp, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
zp = OUTPUT_DIR / "submission_ensemble.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf: zf.write(pp, "prediction.json")
print(f"Saved {zp}")
print(f"{len(predictions)} docs, {sum(len(v) for v in predictions.values())} sentences")

pc = Counter(l for v in predictions.values() for l in v)
ptot, ttot = sum(pc.values()), sum(cnt.values())
print(f"\nPredicted {len(pc)}/{NUM_LABELS} distinct labels")
print(f"{'label':<28}{'pred':>7}{'pred%':>8}{'train%':>9}")
for i in sorted(range(NUM_LABELS), key=lambda i: -cnt.get(i, 0)):
    l = ID2LABEL[i]
    print(f"{l:<28}{pc.get(l,0):>7}{100*pc.get(l,0)/ptot:>7.2f}%{100*cnt.get(i,0)/ttot:>8.2f}%")

json.dump({"backbone": "encoder", "ckpt": str(CKPT_PATH),
           "ckpt_step": ckpt.get("step"), "ckpt_val_loss": ckpt.get("val_loss"),
           "main_best": float(main["best"]), "main_swa": float(main["swa"]),
           "seed_swa": [float(s) for s in swa_list],
           "seed_mean": float(np.mean(swa_list)), "seed_std": float(np.std(swa_list)),
           "ensemble": float(ENS_F1), "pooling_ablation": abl,
           "history": main["hist"]},
          open(OUTPUT_DIR / "encoder_summary.json", "w"), indent=2)
print(f"\nSummary -> {OUTPUT_DIR/'encoder_summary.json'}")
print(f"\nTo compare against the decoder honestly: decoder 3-seed mean +/- std vs this")
print(f"one's {np.mean(swa_list):.4f} +/- {np.std(swa_list):.4f}. A gap under ~0.04 is not a result.")

Saved /content/drive/MyDrive/CS781_Phase3_EncoderHierCRF_outputs/submission_ensemble.zip
10 docs, 2197 sentences

Predicted 11/13 distinct labels
label                          pred   pred%   train%
RatioOfTheDecision              677  30.81%   25.53%
Fact                            453  20.62%   24.33%
PrecedentReliedUpon             445  20.25%   11.55%
ArgumentPetitioner              197   8.97%    9.58%
ArgumentRespondent               56   2.55%    8.00%
Statute                         165   7.51%    6.52%
RulingByLowerCourt              100   4.55%    5.19%
PrecedentNotReliedUpon           48   2.18%    3.38%
RulingByPresentCourt             29   1.32%    1.82%
None                             19   0.86%    1.76%
Dissent                           0   0.00%    1.69%
Issue                             8   0.36%    0.64%
PrecedentOverruled                0   0.00%    0.02%

Summary -> /content/drive/MyDrive/CS781_Phase3_EncoderHierCRF_outputs/encoder_summary.json

To compare against 

In [ ]:
# CELL 11b — submission from the seed-42 model only

predictions = {}
for d in test_docs:
    e = torch.tensor(main["test_e"][d["doc_id"]], dtype=torch.float32, device=DEVICE)
    path = main["crf"].viterbi(e)
    assert len(path) == len(d["sentences"]), d["doc_id"]
    predictions[d["doc_id"]] = [ID2LABEL[int(p)] for p in path]

assert len(predictions) == len(test_docs)
assert all(l in LABEL2ID for v in predictions.values() for l in v)

pp = OUTPUT_DIR / "prediction.json"
json.dump(predictions, open(pp, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
zp = OUTPUT_DIR / "submission_seed42.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pp, "prediction.json")

print(f"Saved {zp}")
print(f"dev F1: best epoch {main['best']:.4f} (ep {main['best_ep']}) | "
      f"plateau avg {main['swa']:.4f}")
print(f"{len(predictions)} docs, {sum(len(v) for v in predictions.values())} sentences")

pc = Counter(l for v in predictions.values() for l in v)
ptot, ttot = sum(pc.values()), sum(cnt.values())
print(f"\nPredicted {len(pc)}/{NUM_LABELS} labels")
for i in sorted(range(NUM_LABELS), key=lambda i: -cnt.get(i, 0)):
    l = ID2LABEL[i]
    print(f"  {l:<26}{pc.get(l,0):>6}{100*pc.get(l,0)/ptot:>7.2f}%"
          f"  train {100*cnt.get(i,0)/ttot:.2f}%")

Saved /content/drive/MyDrive/CS781_Phase3_EncoderHierCRF_outputs/submission_seed42.zip
dev F1: best epoch 0.4680 (ep 25) | plateau avg 0.4516
10 docs, 2197 sentences

Predicted 11/13 labels
  RatioOfTheDecision           558  25.40%  train 25.53%
  Fact                         454  20.66%  train 24.33%
  PrecedentReliedUpon          355  16.16%  train 11.55%
  ArgumentPetitioner           251  11.42%  train 9.58%
  ArgumentRespondent            17   0.77%  train 8.00%
  Statute                      153   6.96%  train 6.52%
  RulingByLowerCourt           142   6.46%  train 5.19%
  PrecedentNotReliedUpon       209   9.51%  train 3.38%
  RulingByPresentCourt          30   1.37%  train 1.82%
  None                          19   0.86%  train 1.76%
  Dissent                        0   0.00%  train 1.69%
  Issue                          9   0.41%  train 0.64%
  PrecedentOverruled             0   0.00%  train 0.02%
